In [1]:
import numpy as np
import torch
import polars as pl

In [2]:
from deep_parity.model import MLP

/Users/dashiell/workspace/deep-parity/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
n = 10
model = MLP(n, 256, 512)

In [4]:
import numpy as np
from deep_parity.boolean_cube import generate_all_binary_arrays, fourier_transform

all_binary_arrays = torch.from_numpy(generate_all_binary_arrays(n)).to(torch.float32)
all_parities = all_binary_arrays.sum(dim=1) % 2

In [5]:
logits, cache = model.run_with_cache(all_binary_arrays)

In [6]:
linear_preacts = cache['hook_linear']

In [42]:
ft = fourier_transform(linear_preacts.T, normalize=True)

In [45]:
ft.T

tensor([[-6.0031e+00, -7.0918e+00,  2.6565e+00,  ...,  3.3194e+00,
          2.6886e-01,  8.4711e+00],
        [-5.0260e-01, -7.4312e-01, -6.5990e-01,  ...,  7.4552e-01,
          8.1625e-01, -7.1411e-01],
        [ 3.5555e+00,  4.9406e-01,  7.1387e-01,  ..., -2.4732e+00,
         -9.2835e-01, -9.5509e-01],
        ...,
        [-8.1025e-08, -1.5600e-08,  1.0710e-08,  ...,  1.4203e-08,
         -1.1316e-07,  5.1223e-09],
        [ 1.1176e-08,  2.9220e-08,  1.4203e-08,  ..., -2.5146e-08,
          2.7474e-08, -1.7742e-07],
        [ 5.9488e-08,  6.6590e-08,  1.6997e-08,  ...,  3.8533e-08,
          1.2259e-07,  2.4796e-08]])

In [9]:
linear_preacts.shape

torch.Size([1024, 512])

In [10]:
2 ** n

1024

In [46]:
ft.shape
linear_df = pl.DataFrame(ft.T.numpy(), schema=[str(i) for i in range(ft.shape[0])])

In [47]:
base_df = pl.DataFrame({
    'bits': all_binary_arrays.numpy(), 
    'parities': all_parities.numpy(), 
    'degree': all_binary_arrays.sum(dim=1).numpy()
}).with_columns(indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1)))

In [48]:
data = pl.concat([base_df, linear_df], how='horizontal')

variable,value
str,f32
"""365""",55.556995
"""61""",45.677078
"""95""",43.195259
"""410""",38.961849
"""419""",74.193802
…,…
"""211""",89.142479
"""207""",70.57119
"""329""",23.832111


In [22]:
power_df = data.select(pl.exclude('bits', 'parities', 'indices')).unpivot(index='degree').group_by('degree', 'variable').agg(pl.col('value').pow(2).sum())

In [32]:
df = power_df.with_columns(pl.col('variable').str.to_integer()).sort(['degree', 'variable']).group_by('degree').agg(pl.col('value').implode())

In [35]:
msg = {int(rec['degree']): rec['value'][0] for rec in df.to_dicts()}

In [24]:
def make_base_parity_dataframe(n):
    all_binary_data = generate_all_binary_arrays(n)
    all_degrees = all_binary_data.sum(axis=1)
    all_parities = all_degrees % 2


    base_df = pl.DataFrame({
        'bits': all_binary_data, 
        'parities': all_parities, 
        'degree': all_degrees
    })
    base_df = base_df.with_columns(
        indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1))
    )
    return base_df


make_base_parity_dataframe(10).head()

bits,parities,degree,indices
"array[u8, 10]",u64,u64,list[u32]
"[0, 0, … 0]",0,0,[]
"[0, 0, … 1]",1,1,[9]
"[0, 0, … 0]",1,1,[8]
"[0, 0, … 1]",0,2,"[8, 9]"
"[0, 0, … 0]",1,1,[7]


In [55]:
def make_base_parity_dataframe(n):
    all_binary_data = generate_all_binary_arrays(n)
    all_degrees = all_binary_data.sum(axis=1)
    all_parities = all_degrees % 2


    base_df = pl.DataFrame({
        'bits': all_binary_data, 
        'parities': all_parities, 
        'degree': all_degrees
    })
    base_df = base_df.with_columns(
        indices=pl.col('bits').arr.to_list().list.eval(pl.arg_where(pl.element() == 1))
    )
    return base_df


def calc_power_contributions(tensor, n, epoch):
    linear_dim = tensor.shape[1]
    base_df = make_base_parity_dataframe(n)
    ft = fourier_transform(tensor)
    linear_df = pl.DataFrame(
        ft.T.numpy(),
        schema=[str(i) for i in range(ft.shape[0])]
    )
    data = pl.concat([base_df, linear_df], how='horizontal')
    total_power = (
        data
        .select(pl.exclude('bits', 'parities', 'indices', 'degree'))
        .unpivot()
        .with_columns(pl.col('variable').str.to_integer())
        .group_by('variable').agg(pl.col('value').pow(2).sum())
        .rename({'value': 'power'})
    )
    power_df = (
        data
        .select(pl.exclude('bits', 'parities', 'indices'))
        .unpivot(index='degree')
        .with_columns(pl.col('variable').str.to_integer())
        .group_by('degree', 'variable').agg(pl.col('value').pow(2).sum())
        .join(total_power, on='variable', how='left')
        .with_columns(pcnt_power = pl.col('value') / pl.col('power'), epoch=pl.lit(epoch))
    )


    return power_df


def fourier_analysis(model, n, epoch):
    bits = generate_all_binary_arrays(n)
    _, cache = model.run_with_cache(bits)
    linear = cache['hook_linear']
    embed_power_df = calc_power_contributions(linear, n, epoch)
    return embed_power_df



In [56]:
power_df = calc_power_contributions(linear_preacts, 10, 0)
power_df.head()

degree,variable,value,power,pcnt_power,epoch
u64,i64,f32,f32,f32,i32
3,959,8.696509,49.025475,0.177388,0
9,311,0.214081,30.881334,0.006932,0
4,352,3.399486,17.287703,0.196642,0
4,832,3.57953,18.943892,0.188954,0
4,343,7.63397,32.438251,0.235339,0


In [2]:
def generate_binary_array_efficient(n):
    # Create an array of all possible numbers for n bits
    numbers = np.arange(2**n, dtype=np.uint32)
    
    # Use broadcasting with a single bitwise operation
    return ((numbers[:, np.newaxis] >> np.arange(n)[::-1]) & 1).astype(np.uint8)


generate_binary_array_efficient(25)

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 1, 0],
       ...,
       [1, 1, 1, ..., 1, 0, 1],
       [1, 1, 1, ..., 1, 1, 0],
       [1, 1, 1, ..., 1, 1, 1]], dtype=uint8)